# 05 — Final results and conclusions

This notebook combines saved results from classification (Notebook 03) and regression (Notebook 04). It does not retrain models.

Run Notebooks 03 and 04 first so the four results CSV files exist.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "results").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / "results"

files = {
    "classification_validation": RESULTS_DIR / "classification_validation_results.csv",
    "classification_test": RESULTS_DIR / "classification_test_results.csv",
    "regression_validation": RESULTS_DIR / "regression_validation_results.csv",
    "regression_test": RESULTS_DIR / "regression_test_results.csv",
}
missing = [name for name, path in files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing}. Run Notebook 03 and Notebook 04 first."
    )

class_val = pd.read_csv(files["classification_validation"])
class_test = pd.read_csv(files["classification_test"])
reg_val = pd.read_csv(files["regression_validation"])
reg_test = pd.read_csv(files["regression_test"])
print("All result files loaded.")

## 1. Classification validation comparison

Use validation results to compare algorithms and feature sets. The test set is not used to choose the model.

In [ ]:
display(class_val.round(4))
class_summary = (class_val.groupby(["Model", "Features"])[
    ["Validation Accuracy", "Balanced Accuracy"]]
    .mean().reset_index().sort_values("Validation Accuracy", ascending=False))
display(class_summary.round(4))

## 2. Classification test results

Compare final model performance with the majority-class baseline for each currency pair.

In [ ]:
display(class_test.round(4))
plot_df = class_test.copy()
plot_df["Label"] = plot_df["Currency Pair"] + " — " + plot_df["Model"]
plt.figure(figsize=(12, 5))
plt.bar(plot_df["Label"], plot_df["Test Accuracy"])
plt.axhline(0.5, linestyle="--", label="50% reference")
plt.ylabel("Test accuracy")
plt.title("Classification test accuracy")
plt.xticks(rotation=55, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Regression results

In [ ]:
display(reg_val.round(6))
display(reg_test.round(6))

reg_plot = reg_test.melt(
    id_vars=["Currency Pair"],
    value_vars=["Test RMSE", "Baseline RMSE"],
    var_name="Method", value_name="RMSE"
)
plt.figure(figsize=(9, 5))
for method in reg_plot["Method"].unique():
    subset = reg_plot[reg_plot["Method"] == method]
    plt.plot(subset["Currency Pair"], subset["RMSE"], marker="o", label=method)
plt.ylabel("RMSE (lower is better)")
plt.title("Regression test RMSE versus baseline")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Report checklist

When writing the conclusion, state:

1. The project predicts next-observation direction (classification) and next-observation log return (regression) for GBP/USD, USD/EUR and USD/JPY.
2. Compare the 7, 15-paper and 15-stationary feature sets using validation results.
3. Report test metrics and compare each model with the appropriate baseline.
4. Negative R² or RMSE worse than the baseline indicates no regression improvement.
5. Directional accuracy alone does not demonstrate profitability.
6. The data source differs from the paper's original Bloomberg data, so this is an alternative-data reproduction attempt, not an exact replication.
7. Notebook 07 adds an illustrative long-or-cash backtest with transaction costs on the test period only; treat it as hypothetical and do not claim that the predictions are profitable.

In [ ]:
print("CLASSIFICATION — best average validation configuration")
display(class_summary.head(1))

print("\nREGRESSION — test performance relative to baseline")
for _, row in reg_test.iterrows():
    print(
        f"{row['Currency Pair']}: R²={row['Test R²']:.4f}; "
        f"model RMSE={row['Test RMSE']:.6f}; "
        f"baseline RMSE={row['Baseline RMSE']:.6f}; "
        f"beats baseline RMSE={row['Test RMSE'] < row['Baseline RMSE']}"
    )